# Chapter 3 — Workflow Patterns for Data Engineering
## AI-Based Data Engineering (Packt)

Two of the five composable workflow patterns from Chapter 3:

1. **Routing** — a classifier dispatches each pipeline alert to the specialist handler that matches its anomaly type. Uses `claude-haiku-4-5` for the narrow taxonomy task, `claude-sonnet-4-5` for deeper analysis.
2. **Evaluator-Optimizer** — a generator produces an output, an evaluator scores it, and the generator refines based on feedback until the quality bar is met.

**Prerequisites:** run `code/setup/opspulse_generator.py --target snowflake` to create the OpsPulse tables. The anomaly-alert SQL cell falls back to synthetic VALUES rows if `OPSPU.PUBLIC.PIPELINE_ALERTS` is absent.

> **External Access Integration required.** This notebook calls the Anthropic API from Snowflake. Your admin must:
> 1. Create an External Access Integration for `api.anthropic.com`
> 2. Set `ANTHROPIC_API_KEY` as a Snowflake Secret and attach it to this notebook
>
> Without this, the `anthropic.Anthropic()` calls will fail. See [Snowflake EAI docs](https://docs.snowflake.com/en/developer-guide/external-network-access/creating-using-external-network-access).


In [ ]:
from snowflake.snowpark.context import get_active_session
import anthropic
import json
from enum import Enum
from pydantic import BaseModel, Field

session = get_active_session()
client  = anthropic.Anthropic()
print("Session and Anthropic client ready.")

In [ ]:
%%sql -r anomaly_alerts
-- PIPELINE_ALERTS is created by the OpsPulse generator.
-- Using a VALUES fallback so this cell runs even without the generator.
WITH synthetic_alerts (alert_id, table_fqn, alert_type, null_rate, minutes_late) AS (
    SELECT column1::VARCHAR, column2::VARCHAR, column3::VARCHAR,
           column4::FLOAT, column5::INT
    FROM VALUES
        ('ALT001', 'OPSPU.MARTS.FCT_SALES',            'volume_drop',      0.0,   0),
        ('ALT002', 'OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS', 'freshness_breach', 0.0, 187),
        ('ALT003', 'OPSPU.RAW.IOT_EVENTS',             'quality_failure',  0.041, 0),
        ('ALT004', 'OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS', 'schema_change',    0.0,   0),
        ('ALT005', 'OPSPU.MARTS.FCT_REVENUE',          'volume_drop',      0.0,   0)
)
SELECT * FROM synthetic_alerts;
-- To use real data instead:
-- SELECT alert_id, table_fqn, alert_type, null_rate, minutes_late
-- FROM OPSPU.PUBLIC.PIPELINE_ALERTS LIMIT 20;

## Pattern 1: Routing

The routing pattern uses a cheap, fast classifier to inspect each alert and dispatch it to the specialist handler that matches its anomaly type. The classifier returns a `TriageClassification` with:

- `anomaly_type` — one of five canonical types
- `confidence` — 0.0–1.0; below 0.70 the alert escalates to the fallback handler
- `primary_signal` — the data point that drove the classification
- `recommended_action` — first step for the on-call engineer

**Key design choice:** the classifier uses `claude-haiku-4-5` (fast, cheap, narrow taxonomy task); specialist handlers use `claude-sonnet-4-5` (quality analysis). You only pay sonnet prices when the input is worth analyzing.

In [ ]:
# ── Types, classifier, handlers, dispatcher ────────────────────────────────

class AnomalyType(str, Enum):
    VOLUME_DROP      = 'volume_drop'
    SCHEMA_CHANGE    = 'schema_change'
    FRESHNESS_BREACH = 'freshness_breach'
    QUALITY_FAILURE  = 'quality_failure'
    LINEAGE_BREAK    = 'lineage_break'
    UNKNOWN          = 'unknown'


class TriageClassification(BaseModel):
    anomaly_type:       AnomalyType
    confidence:         float = Field(ge=0.0, le=1.0)
    primary_signal:     str
    recommended_action: str


CLASSIFIER_SYSTEM = (
    'You are a data reliability engineer triaging data pipeline alerts. '
    'Classify anomalies into canonical types to route them to the correct '
    'specialist handler. Never speculate beyond what the alert data supports. '
    'When uncertain, use anomaly_type=unknown and set confidence below 0.5.'
)


def classify_alert(alert_payload: dict) -> TriageClassification:
    """Classifier step: uses claude-haiku-4-5 (fast, cheap, narrow task)."""
    response = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=300,
        system=CLASSIFIER_SYSTEM,
        tools=[{
            'name': 'classify',
            'description': 'Classify the anomaly type.',
            'input_schema': TriageClassification.model_json_schema(),
        }],
        tool_choice={'type': 'tool', 'name': 'classify'},
        messages=[{'role': 'user', 'content': (
            f'Alert payload: {json.dumps(alert_payload)}\n\n'
            'Classification rules:\n'
            '- volume_drop: row count or record volume fell below threshold\n'
            '- schema_change: column added, removed, renamed, or type changed\n'
            '- freshness_breach: data not updated within the expected window\n'
            '- quality_failure: null rate, uniqueness, or referential integrity breach\n'
            '- lineage_break: upstream table or model failed to produce output\n'
            '- unknown: insufficient data to classify confidently'
        )}],
    )
    tool_call = next(b for b in response.content if b.type == 'tool_use')
    return TriageClassification(**tool_call.input)


def handle_volume_drop(alert: dict, clf: TriageClassification) -> dict:
    response = client.messages.create(
        model='claude-sonnet-4-5', max_tokens=500,
        system=(
            'You are a data reliability engineer investigating row count drops. '
            'Identify the most likely root cause and provide specific remediation '
            'steps for a Snowflake/Airflow stack.'
        ),
        messages=[{'role': 'user', 'content': (
            f'Alert: {json.dumps(alert)}\n'
            f'Classification: {clf.model_dump_json()}\n\n'
            'Provide: root cause hypothesis, one verification query, remediation steps.'
        )}],
    )
    return {'handler': 'volume_drop', 'analysis': response.content[0].text}


def handle_quality_failure(alert: dict, clf: TriageClassification) -> dict:
    response = client.messages.create(
        model='claude-sonnet-4-5', max_tokens=500,
        system=(
            'You are a data reliability engineer investigating data quality failures. '
            'Analyze null rate, uniqueness, or referential integrity issues and '
            'recommend remediation.'
        ),
        messages=[{'role': 'user', 'content': (
            f'Alert: {json.dumps(alert)}\n\n'
            'Provide: quality dimension failing, root cause, fix.'
        )}],
    )
    return {'handler': 'quality_failure', 'analysis': response.content[0].text}


def handle_unknown(alert: dict, clf: TriageClassification) -> dict:
    return {
        'handler': 'escalation',
        'analysis': 'Confidence below threshold. Escalated to on-call engineer.',
        'classification': clf.model_dump(),
    }


ROUTE_MAP = {
    AnomalyType.VOLUME_DROP:      handle_volume_drop,
    AnomalyType.QUALITY_FAILURE:  handle_quality_failure,
    AnomalyType.SCHEMA_CHANGE:    handle_unknown,
    AnomalyType.FRESHNESS_BREACH: handle_unknown,
    AnomalyType.LINEAGE_BREAK:    handle_unknown,
    AnomalyType.UNKNOWN:          handle_unknown,
}


def triage_alert(alert_payload: dict) -> dict:
    """Full routing pipeline: classify → dispatch → analyze."""
    clf = classify_alert(alert_payload)
    if clf.confidence < 0.70:
        return handle_unknown(alert_payload, clf)
    handler = ROUTE_MAP.get(clf.anomaly_type, handle_unknown)
    result  = handler(alert_payload, clf)
    result['classification'] = clf.model_dump()
    return result


print("Routing functions defined.")

In [ ]:
# ── Test the router with sample alerts ─────────────────────────────────────

test_alerts = [
    {
        'table': 'OPSPU.MARTS.FCT_SALES',
        'check': 'row_count',
        'row_count_today': 4_200,
        'row_count_yesterday': 9_800,
        'threshold_pct': 20,
    },
    {
        'table': 'OPSPU.RAW.IOT_EVENTS',
        'check': 'null_rate',
        'column': 'device_timestamp',
        'null_rate': 0.041,
        'threshold': 0.02,
    },
]

for alert in test_alerts:
    print(f"\n{'='*60}")
    print(f"Alert: {alert['table']} / {alert['check']}")
    result = triage_alert(alert)
    clf    = result['classification']
    print(f"  Type:       {clf['anomaly_type']}")
    print(f"  Confidence: {clf['confidence']:.0%}")
    print(f"  Signal:     {clf['primary_signal']}")
    print(f"  Handler:    {result['handler']}")
    if result['handler'] != 'escalation':
        print(f"  Analysis (preview):\n{result.get('analysis', '')[:400]}")

## Pattern 2: Evaluator-Optimizer

The evaluator-optimizer terminates when an output meets the quality bar — or when the iteration cap is reached. Three steps:

1. **Generate** — produce an initial output (a column description here)
2. **Evaluate** — score the output against explicit criteria; return `passes` + `feedback`
3. **Regenerate** — if `passes == False`, pass the feedback to the generator and retry

**Why it matters for data engineering:** column descriptions that fail the quality bar produce misleading catalog entries. An LLM-as-judge inside the generation loop catches these failures before they reach production, without requiring a human reviewer on every column.

In [ ]:
# ── Evaluator-Optimizer: column documentation loop ─────────────────────────

class ColumnEval(BaseModel):
    score:    int  = Field(ge=1, le=5, description='Quality score 1-5')
    passes:   bool = Field(description='True if score >= 4')
    feedback: str  = Field(description='One sentence of improvement guidance')


def generate_description(column_name: str, data_type: str, table_name: str) -> str:
    response = client.messages.create(
        model='claude-haiku-4-5', max_tokens=100,
        messages=[{'role': 'user', 'content': (
            f"Write a one-sentence business description for column '{column_name}' "
            f"({data_type}) in table '{table_name}'. No jargon. No quotation marks."
        )}],
    )
    return response.content[0].text.strip()


def evaluate_description(column_name: str, description: str) -> ColumnEval:
    response = client.messages.create(
        model='claude-haiku-4-5', max_tokens=150,
        tools=[{
            'name': 'evaluate',
            'description': 'Score a column description.',
            'input_schema': ColumnEval.model_json_schema(),
        }],
        tool_choice={'type': 'tool', 'name': 'evaluate'},
        messages=[{'role': 'user', 'content': (
            f"Score this description for '{column_name}': \"{description}\"\n"
            'Criteria: clear, business-focused, no jargon, states what the column contains.\n'
            'Score: 5=excellent, 4=passes, 3=borderline, 1-2=fails.'
        )}],
    )
    tool_call = next(b for b in response.content if b.type == 'tool_use')
    return ColumnEval(**tool_call.input)


def regenerate_with_feedback(
    column_name: str, data_type: str, table_name: str,
    prior: str, feedback: str,
) -> str:
    response = client.messages.create(
        model='claude-haiku-4-5', max_tokens=100,
        messages=[{'role': 'user', 'content': (
            'Rewrite this description based on feedback.\n'
            f"Column: '{column_name}' ({data_type}) in '{table_name}'\n"
            f'Prior:    {prior}\n'
            f'Feedback: {feedback}\n'
            'Write an improved one-sentence description. No quotation marks.'
        )}],
    )
    return response.content[0].text.strip()


def evaluator_optimizer(
    column_name: str, data_type: str, table_name: str,
    max_iterations: int = 3,
) -> str:
    description = generate_description(column_name, data_type, table_name)
    for i in range(max_iterations):
        ev = evaluate_description(column_name, description)
        print(f'  Iter {i+1}: score={ev.score}/5  passes={ev.passes}')
        print(f'    {description}')
        if ev.passes:
            print('    Accepted.')
            return description
        print(f'    Feedback: {ev.feedback}')
        description = regenerate_with_feedback(
            column_name, data_type, table_name, description, ev.feedback
        )
    print('  Max iterations reached.')
    return description


# Run on three columns from fct_active_customers
columns_to_document = [
    ('customer_id',      'VARCHAR',      'FCT_ACTIVE_CUSTOMERS'),
    ('region_code',      'VARCHAR',      'FCT_ACTIVE_CUSTOMERS'),
    ('total_orders_30d', 'NUMBER(10,0)', 'FCT_ACTIVE_CUSTOMERS'),
]

print('Evaluator-Optimizer column documentation run\n')
final_descriptions = {}
for col_name, col_type, tbl in columns_to_document:
    print(f'Column: {col_name}')
    final_descriptions[col_name] = evaluator_optimizer(col_name, col_type, tbl)
    print()

print('Final descriptions:')
for col, desc in final_descriptions.items():
    print(f'  {col}: {desc}')

In [ ]:
%%sql -r schema_context
-- Schema context for the generated descriptions above
SELECT
    column_name,
    data_type,
    COALESCE(comment, '[no description]') AS comment
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
  AND table_name   = 'FCT_ACTIVE_CUSTOMERS'
ORDER BY ordinal_position;

## Summary

| Pattern | When to use | Cost profile |
|---|---|---|
| **Routing** | Heterogeneous inputs requiring specialist handling | haiku triage, sonnet only when warranted |
| **Evaluator-Optimizer** | Quality bar on generated outputs; human review too slow | 2–3× single-pass cost, avoids rework |

The three remaining patterns from Chapter 3 — Parallelization, Orchestrator-Workers, and Agent loops — build on the same primitives. See `code/ch03_workflow_patterns/` for the full implementations.